# AI Eval 101 — run a real evaluation yourself (SimpleQA)

Companion notebook to the **BankBench-MY / AI Eval 101** deck. Ten minutes, start to finish.

[SimpleQA](https://arxiv.org/abs/2411.04368) (Wei et al., 2024) is a benchmark of **4,326 short,
fact-seeking questions**, each with one verifiable answer. It is a good first eval because you can
run three questions instead of 4,326, and because its scoring rule is unusually honest: it
distinguishes *"doesn't know"* from *"doesn't know and answered anyway"*.

We use **Inspect**, the evaluation framework from the **UK AI Safety Institute**, and its
[`inspect_evals`](https://github.com/UKGovernmentBEIS/inspect_evals) implementation of SimpleQA.

### Before you start

**Colab path (no install):** add an API key to Colab's **Secrets** panel — the key icon 🔑 in the
left sidebar — so it never sits in a cell you might share. Name it `OPENAI_API_KEY`.
Steps 1–2 below need **no key at all**, so you can read this whole notebook for free.

**VS Code / local path:** `pip install inspect-ai inspect-evals`, then install the
**Inspect AI extension** so `logs/*.eval` opens as a clickable transcript in the editor.
Everything below works identically — Colab-only cells are marked as such.

> **If the first import fails in Colab:** Runtime → *Restart session*, then re-run from the top.
> Colab ships its own versions of some libraries and a restart settles it.

## Step 1 · Watch the whole pipeline move, with no API key

`mockllm/model` is Inspect's built-in fake model. It makes **no network calls and costs nothing**,
so you can see the entire Task → dataset → solver → scorer → log shape before you spend a cent.

Note the shape of the task object: `simpleqa()` is a `@task` function, and calling it returns a
`Task` with three parts — a **dataset**, a **solver**, and a **scorer**. That is the whole framework.

In [ ]:
# `ipywidgets` is required: Inspect detects that it is running inside a notebook and
# needs it for progress display. Colab normally has it, but pinning it here means the
# notebook works on a bare kernel too.
%pip install -q inspect-ai inspect-evals ipywidgets

In [ ]:
import glob, os

from inspect_ai import eval as inspect_eval
from inspect_evals.simpleqa import simpleqa

LOG_DIR = "logs-simpleqa-101"
os.makedirs(LOG_DIR, exist_ok=True)

# scorer="original" = the paper's A/B/C string-matching scorer, which reports the full
# six-field breakdown. The package DEFAULT ("tool") reports a single `accuracy` instead,
# and prints nan when nothing could be graded -- which is itself a useful thing to see.
task = simpleqa(scorer="original")

logs = inspect_eval(
    task,
    model="mockllm/model",   # fake model: no key, no network, no cost
    limit=5,                 # 5 of the 4,326 questions
    log_dir=LOG_DIR,
    display="none",          # so the notebook output stays readable
)

log = logs[0]
print("status :", log.status)
print("task   :", log.eval.task)
print("model  :", log.eval.model)
print("samples:", len(log.samples))

print("\n-- the result: six numbers, not one --")
for sc in log.results.scores:
    print("scorer:", sc.name)
    for k, v in sc.metrics.items():
        print(f"   {k:26s} {v.value}")

### How to read those six numbers

| Field | What it means |
|---|---|
| `correct` | share of samples graded **A** |
| `incorrect` | share graded **B** |
| `not_attempted` | share graded **C** — a refusal, *or* a reply the grader could not parse |
| `attempted` | `1 - not_attempted` |
| `correct_given_attempted` | `correct / (correct + incorrect)` — precision **when it chose to answer** |
| `f_score` | harmonic mean of `correct` and `correct_given_attempted` |

**Why `f_score` and not just accuracy?** A model that answers everything confidently and is wrong
some of the time gets punished **twice** — once for the wrong answer, once for the drop in
`correct_given_attempted`. Abstaining is not a free pass either. SimpleQA is deliberately built to
score the dangerous case (*confidently wrong*) worse than the safe one (*silent*).

In this run the mock replies are gibberish, so the grader could not parse them and graded every
sample C. Hence `not_attempted = 1.0`. **Those values are meaningless as a measurement** — the
point is that a grading failure shows up as `not_attempted`, visibly, instead of being silently
folded into "wrong". That behaviour is the reason we trust this scorer enough to build on it.

## Step 2 · See who actually does the grading

This is the part people miss. The grade did **not** come from the model under test. Inspect asks a
**second model** to read the question, the gold answer, and the agent's answer, and return a letter.

Look at the brand names in the examples below — the grader is a different model in a different lab.
That separation is deliberate, and it is exactly why the evaluation **quality scorecard** puts such
weight on the *Correctness* dimension (inter-rater agreement) and why the discipline is: never let
one model draft the answer, judge it, and grade itself.

In [ ]:
from inspect_evals.simpleqa.grader_templates import SIMPLEQA_GRADER_TEMPLATE

# This is the literal prompt sent to the grader model — a second model, not the one under test.
print(SIMPLEQA_GRADER_TEMPLATE[:2600])

## Step 3 · A real number (this is the part that needs a key)

Same task, same scorer, a real model. We keep `limit=20` — 20 questions plus 20 grader calls is a
few cents. Drop the `limit` and this becomes 4,326 questions *plus* a grader call for each, which
is why the flag matters.

In [ ]:
import os

# Colab: read the key from the Secrets panel (key icon in the left sidebar).
# Locally: just `export OPENAI_API_KEY=...` in your shell and delete this whole cell.
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("key loaded from Colab secrets")
except Exception:
    print("(no Colab secret found -- that is fine)")

HAVE_KEY = bool(os.environ.get("OPENAI_API_KEY"))
print("key present:", HAVE_KEY)
if not HAVE_KEY:
    print()
    print("Steps 1, 2 and 4 above and below already showed you the whole mechanism for free.")
    print("To turn the mock into a real measurement, either:")
    print("   - Colab: add OPENAI_API_KEY in the Secrets panel (key icon, left sidebar), or")
    print("   - local: os.environ['OPENAI_API_KEY'] = 'sk-...'")
    print("Then re-run this cell and the next one.")

In [ ]:
# Swap the model for any Inspect provider you have a key for, e.g.
#   anthropic/claude-3-5-haiku-latest   |   google/gemini-2.0-flash
#   together/meta-llama/Llama-3.3-70B-Instruct-Turbo   |   openrouter/deepseek/deepseek-chat
MODEL = "openai/gpt-4o-mini"

if not HAVE_KEY:
    # Degrade gracefully instead of crashing, so "Run all" completes for a keyless reader.
    print("Skipping the live run (no API key). Falling back to the mock log from Step 1.")
    print("Everything below still works -- it just reports the mock numbers.")
    real = log
else:
    real_logs = inspect_eval(
        simpleqa(scorer="original"),
        model=MODEL,
        limit=20,
        log_dir=LOG_DIR,
        display="none",
    )
    real = real_logs[0]

print("model  :", real.eval.model)
print("status :", real.status)
print("samples:", len(real.samples))
print()
for sc in real.results.scores:
    print("scorer:", sc.name)
    for k, v in sc.metrics.items():
        print(f"   {k:26s} {v.value}")

## Step 4 · Read it per sample, never in aggregate

A total tells you *that* something is wrong. Only the per-sample rows tell you *what*. This is the
cell that matters — reading individual answers is how we found refusals being scored as wrong in
our own suite, a bug that was invisible in every summary number.

Every run is written to a `.eval` file. Nothing is held in memory, so you can go back and re-read
any run without re-running it.

In [ ]:
from inspect_ai.log import read_eval_log

newest = max(glob.glob(os.path.join(LOG_DIR, "*.eval")), key=os.path.getmtime)
print("reading:", os.path.basename(newest), "\n")

run = read_eval_log(newest)

for s in run.samples:
    print("Q:", (s.input or "").replace("\n", " ")[:88])
    print("   answered:", (s.output.completion or "").replace("\n", " ")[:88])
    for name, sc in (s.scores or {}).items():
        print("   graded  :", sc.value)
    print()

In [ ]:
# Recompute f_score from the raw counts, so the number is not a black box.
m = {k: v.value for k, v in run.results.scores[0].metrics.items()}
c, inc, na = m["correct"], m["incorrect"], m["not_attempted"]

cga = c / (c + inc) if (c + inc) > 0 else 0.0          # precision when it answered
f   = 2 * c * cga / (c + cga) if (c + cga) > 0 else 0.0  # harmonic mean

print(f"correct                   {c:.3f}")
print(f"incorrect                 {inc:.3f}")
print(f"not_attempted             {na:.3f}   <- abstentions + unparseable replies")
print(f"attempted                 {1 - na:.3f}")
print(f"correct_given_attempted   {cga:.3f}   (recomputed)")
print(f"f_score                   {f:.3f}   (recomputed; reported: {m['f_score']})")

print()
print("With only", int(c + inc), "graded answers from", len(run.samples),
      "questions, this is an EXPLORATORY result.")
print("Per the quality scorecard it would be graded Category E -- useful for a first look,")
print("not for anything anyone makes a decision on. Raise `limit` and add repeats first.")

## Step 5 · (Colab only) open the log viewer

The best way to read an eval is Inspect's own viewer, which renders every sample as a transcript.
Locally it is just `inspect view`. In Colab we run it and forward the port so the notebook can
frame it.

**This is the one cell that only works inside Colab** — it is untested anywhere else, so if it
misbehaves just skip it; step 4 already shows you the same information and
[inspect.ai-safety-institute.org.uk](https://inspect.ai-safety-institute.org.uk) documents the
viewer properly.

In [ ]:
import subprocess, time

# Colab-only: serve the viewer on the notebook's port proxy.
try:
    from google.colab import output
    proc = subprocess.Popen(
        ["inspect", "view", "--host", "0.0.0.0", "--port", "8080", "--log-dir", LOG_DIR],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    time.sleep(6)
    output.serve_kernel_port_as_iframe(8080, height=720)
except ImportError:
    print("Not in Colab. Locally just run:  inspect view --log-dir", LOG_DIR)

## Where to go next

You have now run a real published benchmark on the Inspect framework, read its six-field result,
and seen that a **second model** produced the grades. That is the whole shape of an evaluation.

- **The framework** — [Inspect, UK AI Safety Institute](https://inspect.ai-safety-institute.org.uk)
- **The evals** — [`inspect_evals`](https://github.com/UKGovernmentBEIS/inspect_evals), ~100 benchmarks
- **SimpleQA** — [arXiv 2411.04368](https://arxiv.org/abs/2411.04368); the cleaned-up
  [SimpleQA Verified](https://arxiv.org/abs/2509.07968) is 1,000 prompts with less label noise
- **The quality scorecard** —
  [AI Standards Lab: A Scorecard for the Quality of AI Evaluations](https://aistandardslab.org/a-scorecard-for-the-quality-of-ai-evaluations/)

**Two things to try before you close this notebook**

1. Run step 3 twice at the same `limit` and compare. If the exact same questions score differently
   between runs, you have measured *noise*, not capability — which is why the scorecard wants
   repeats (`N>=3`) per cell before a result means anything.
2. Run step 3 with `limit=20`, then `limit=200`. Watch `f_score` move. A benchmark whose score
   depends on how many questions you asked has a problem worth writing down.

> **What this notebook is not:** a verdict on any model or product. It is one trial run on a
> 20-question sample with no repeats — an indicator, and nothing more.